# 🧪 Cuaderno 02 — Laboratorios y Prácticas de Métodos

**Métodos Numéricos · Unidad 3: Sistemas de Ecuaciones Lineales**

Laboratorios interactivos y preguntas sobre los **4 métodos**. La **teoría completa**
está en los markdown (enlazados en cada sección); aquí solo vienes a **practicar y
comprobar** que entiendes.

> 📖 **Teoría (markdown):** [Eliminación Gaussiana](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/main/docs/metodos/01_eliminacion_gaussiana.md) · [LU](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/main/docs/metodos/02_descomposicion_lu.md) · [Gauss-Seidel](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/main/docs/metodos/03_metodos_iterativos.md) · [Thomas/Cholesky](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/main/docs/metodos/04_matrices_especiales.md)


In [ ]:
#@title ⚙️ Entorno y conexión al backend (ejecuta PRIMERO)
import os, json, urllib.request
try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

ARCHIVO = "robomat.py"
URL = "https://raw.githubusercontent.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/main/lib/robomat.py"


def _buscar_robomat(nombre=ARCHIVO, saltos=6):
    """Busca robomat.py junto al notebook, en la carpeta actual y en lib/ de los padres."""
    bases = [os.getcwd()]
    ruta_nb = globals().get("__vsc_ipynb_file__")      # VS Code
    if ruta_nb:
        bases.append(os.path.dirname(os.path.abspath(ruta_nb)))
    for base in list(bases):
        b = base
        for _ in range(saltos):
            for cand in (os.path.join(b, nombre), os.path.join(b, "lib", nombre)):
                if os.path.exists(cand):
                    return os.path.abspath(cand)
            padre = os.path.dirname(b)
            if padre == b:
                break
            b = padre
    return None


ruta_robomat = _buscar_robomat()
if ruta_robomat is None:
    print("robomat.py no está en disco; descargando del repositorio...")
    urllib.request.urlretrieve(URL, ARCHIVO)
    ruta_robomat = os.path.abspath(ARCHIVO)

exec(open(ruta_robomat, encoding="utf-8").read(), globals())
print(f"✅ Entorno cargado desde: {ruta_robomat}")

import numpy as np
try:
    from ipywidgets import interact, IntSlider
except ImportError:
    interact = None

# ===== Conexión al backend (Apps Script) =====
APPS_SCRIPT_URL = ""   #@param {type:"string"}
WEBHOOK_TOKEN   = ""   #@param {type:"string"}
NC = ""                #@param {type:"string"}
BACKEND_LISTO = bool(APPS_SCRIPT_URL.strip() and WEBHOOK_TOKEN.strip())

def _post(accion, **campos):
    payload = {"accion": accion, "token": WEBHOOK_TOKEN, **campos}
    req = urllib.request.Request(
        APPS_SCRIPT_URL, data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=30) as r:
        resp = json.loads(r.read().decode("utf-8"))
    if resp.get("status") != "ok":
        raise RuntimeError(resp.get("message", "Error del backend"))
    return resp.get("data", {})

print("✅ Entorno:", "Google Colab" if ES_COLAB else "local")
print("🔌 Backend:", "CONECTADO" if BACKEND_LISTO else "sin configurar (solo práctica local)")


## 1. Eliminación Gaussiana · laboratorio interactivo

Mueve el deslizador para ver la matriz aumentada tras cada **pivote** (eliminación hacia adelante).
📖 [Teoría completa](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/main/docs/metodos/01_eliminacion_gaussiana.md)


In [ ]:
A = np.array([[3., -0.1, -0.2], [0.1, 7., -0.3], [0.3, -0.2, 10.]])
b = np.array([7.85, -19.3, 71.4])

def matriz_tras_pivote(k):
    Ab = np.c_[A.copy(), b.copy()]
    n = len(b)
    for j in range(min(k, n-1)):
        p = j + np.argmax(np.abs(Ab[j:, j]))
        if p != j: Ab[[j, p]] = Ab[[p, j]]
        for i in range(j+1, n):
            m = Ab[i, j] / Ab[j, j]
            Ab[i, j:] = Ab[i, j:] - m * Ab[j, j:]
    print(Ab)

interact(matriz_tras_pivote, k=IntSlider(min=0, max=2, step=1, value=0, description="pivote"))


### ✏️ Pregunta 1 (teórica)
¿Qué problema(s) resuelve el **pivoteo parcial**?


In [ ]:
res_1 = "b"  #@param ["a", "b", "c", "d"]
#@markdown ¿Qué problema(s) resuelve el **pivoteo parcial**?
#@markdown - a) acelera la convergencia
#@markdown - b) evita división entre cero y amplificación de error
#@markdown - c) reduce la memoria
#@markdown - d) no resuelve nada
CORRECTA_1 = "b"
print("✅ Correcto" if res_1 == CORRECTA_1 else "❌ Revisa la sección de pivoteo en el markdown")


### 🔢 Pregunta 2 (numérica)
Resuelve a mano este sistema $2\times2$ y escribe $x_1$:

$$\begin{bmatrix} 2 & 1 \\ 1 & 3 \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \end{bmatrix} = \begin{bmatrix} 4 \\ 7 \end{bmatrix}$$


In [ ]:
res_2 = 1.0  #@param {type:"number"}
#@markdown Resuelve $\\begin{bmatrix}2&1\\\\1&3\\end{bmatrix}\\begin{bmatrix}x_1\\\\x_2\\end{bmatrix}=\\begin{bmatrix}4\\\\7\\end{bmatrix}$ y escribe $x_1$.
_A2 = np.array([[2., 1.], [1., 3.]]); _b2 = np.array([4., 7.])
x1_correcto = float(np.linalg.solve(_A2, _b2)[0])
print(f"✅ Correcto, x1 = {x1_correcto}" if abs(res_2 - x1_correcto) < 1e-6
      else "❌ Incorrecto. Pista: despeja x2 de la 1ª y sustituye en la 2ª.")


## 2. Descomposición LU · laboratorio interactivo

Factoriza una matriz y verifica $\mathbf{A}=\mathbf{L}\mathbf{U}$.
📖 [Teoría completa](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/main/docs/metodos/02_descomposicion_lu.md)


In [ ]:
def descomponer_lu(A):
    A = np.array(A, dtype=float).copy(); n = A.shape[0]
    L = np.eye(n); U = np.zeros((n,n))
    for k in range(n):
        U[k, k:] = A[k, k:]
        for i in range(k+1, n):
            L[i,k] = A[i,k]/U[k,k]
            A[i,k:] = A[i,k:] - L[i,k]*U[k,k:]
    return L, U

L, U = descomponer_lu(np.array([[3.,-0.1,-0.2],[0.1,7.,-0.3],[0.3,-0.2,10.]]))
print("L =\n", L)
print("U =\n", U)
print("¿A == L@U?", np.allclose(np.array([[3.,-0.1,-0.2],[0.1,7.,-0.3],[0.3,-0.2,10.]]), L@U))


### ✏️ Pregunta 3 (teórica)
¿Cuándo conviene usar **LU** en lugar de eliminación gaussiana simple?


In [ ]:
res_3 = "b"  #@param ["a", "b", "c", "d"]
#@markdown ¿Cuándo conviene usar **LU** en lugar de eliminación gaussiana simple?
#@markdown - a) siempre, es más rápido
#@markdown - b) cuando hay que resolver varios vectores b con la misma A
#@markdown - c) solo para matrices tridiagonales
#@markdown - d) nunca
CORRECTA_3 = "b"
print("✅ Correcto" if res_3 == CORRECTA_3 else "❌ Revisa la motivación de LU en el markdown")


## 3. Gauss-Seidel · convergencia interactiva

Mueve el deslizador para ver cómo converge la solución. 📖 [Teoría completa](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/main/docs/metodos/03_metodos_iterativos.md)


In [ ]:
def gs_hist(iteraciones):
    A = np.array([[3.,-0.1,-0.2],[0.1,7.,-0.3],[0.3,-0.2,10.]])
    b = np.array([7.85,-19.3,71.4]); n = len(b)
    x = np.zeros(n); xref = np.linalg.solve(A,b)
    err = []
    for _ in range(iteraciones):
        xo = x.copy()
        for i in range(n):
            s = np.dot(A[i,:i],x[:i]) + np.dot(A[i,i+1:],xo[i+1:])
            x[i] = (b[i]-s)/A[i,i]
        err.append(np.max(np.abs(x-xref)))
    fig, ax = plt.subplots()
    ax.semilogy(range(1, iteraciones+1), err, marker="o")
    ax.set_xlabel("iteración"); ax.set_ylabel("error (log)"); ax.set_title("Convergencia de Gauss-Seidel")
    plt.show()

interact(gs_hist, iteraciones=IntSlider(min=1, max=15, step=1, value=1))


### ✏️ Pregunta 4 (teórica)
¿Qué condición **garantiza** la convergencia de Gauss-Seidel?


In [ ]:
res_4 = "c"  #@param ["a", "b", "c", "d"]
#@markdown ¿Qué condición **garantiza** la convergencia de Gauss-Seidel?
#@markdown - a) matriz simétrica
#@markdown - b) matriz densa
#@markdown - c) dominancia diagonal estricta
#@markdown - d) matriz con ceros en la diagonal
CORRECTA_4 = "c"
print("✅ Correcto" if res_4 == CORRECTA_4 else "❌ Revisa la condición de dominancia diagonal")


## 4. Algoritmo de Thomas · laboratorio

Resuelve un sistema tridiagonal en $O(n)$.
📖 [Teoría completa](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/main/docs/metodos/04_matrices_especiales.md)


In [ ]:
def thomas(e, f, g, r):
    n = len(f); e=np.array(e,float); f=np.array(f,float).copy()
    g=np.array(g,float); r=np.array(r,float).copy()
    for k in range(1,n):
        fac=e[k]/f[k-1]; f[k]-=fac*g[k-1]; r[k]-=fac*r[k-1]
    x=np.zeros(n); x[-1]=r[-1]/f[-1]
    for k in range(n-2,-1,-1): x[k]=(r[k]-g[k]*x[k+1])/f[k]
    return x

f=[2.,2.,2.]; e=[0.,-1.,-1.]; g=[-1.,-1.,0.]; r=[1.,0.,1.]
print("Solución (Thomas):", thomas(e,f,g,r))


### ✏️ Pregunta 5 (teórica)
¿Cuál es la **complejidad** del algoritmo de Thomas para un sistema $n\times n$ tridiagonal?


In [ ]:
res_5 = "a"  #@param ["a", "b", "c", "d"]
#@markdown ¿Cuál es la **complejidad** del algoritmo de Thomas para un sistema $n\\times n$ tridiagonal?
#@markdown - a) O(n)
#@markdown - b) O(n²)
#@markdown - c) O(n³)
#@markdown - d) O(log n)
CORRECTA_5 = "a"
print("✅ Correcto" if res_5 == CORRECTA_5 else "❌ Revisa la complejidad en el markdown")


## 5. Python vs MATLAB (comparación teórica)

| Operación | Python (NumPy/SciPy) | MATLAB |
|---|---|---|
| Resolver $\mathbf{A}\mathbf{x}=\mathbf{b}$ | `np.linalg.solve(A, b)` | `A \\ b` |
| Descomposición LU | `scipy.linalg.lu(A)` | `[L,U,P] = lu(A)` |
| Determinante / inversa | `np.linalg.det` / `inv` | `det(A)` / `inv(A)` |
| Número de condición | `np.linalg.cond(A)` | `cond(A)` |
| Factorización Cholesky | `np.linalg.cholesky(A)` | `chol(A)` |

**En síntesis:** la **barra invertida** de MATLAB (`A \\ b`) elige automáticamente el mejor solver,
mientras que en Python debes elegir la función. Ambos usan las mismas rutinas LAPACK/BLAS por debajo.


#@title 📤 Enviar calificación (máximo 2 intentos)
PESOS = [1, 1, 1, 1, 1]   # un punto por pregunta (ajústalos si quieres)

def calcular():
    """Devuelve la lista de aciertos (1/0) por pregunta."""
    r = []
    r.append(1 if globals().get("res_1") == globals().get("CORRECTA_1") else 0)
    _x1c = float(np.linalg.solve(np.array([[2.,1.],[1.,3.]]), np.array([4.,7.]))[0])
    r.append(1 if abs(globals().get("res_2", 0) - _x1c) < 1e-6 else 0)
    r.append(1 if globals().get("res_3") == globals().get("CORRECTA_3") else 0)
    r.append(1 if globals().get("res_4") == globals().get("CORRECTA_4") else 0)
    r.append(1 if globals().get("res_5") == globals().get("CORRECTA_5") else 0)
    return r

aciertos = calcular()
print("Respuestas:", ["✅" if a else "❌" for a in aciertos])
print("Puntaje local:", sum(aciertos), "/", len(aciertos))

if not BACKEND_LISTO:
    print("\nℹ️ Backend sin configurar: tu puntaje NO se envió.")
elif not NC.strip():
    print("\n⚠️ Escribe tu NC en la celda de entorno para poder enviar.")
else:
    try:
        res = _post("guardar", NC=NC, tarea="U3_02_Laboratorios",
                    respuestas=aciertos, pesos=PESOS, maxPuntos=1)
        print(f"\n✅ Enviado. Intento {res.get('intento')} · Total {res.get('total', 0):.1f}/100")
    except Exception as e:
        print("\n⛔ No se pudo enviar:", e)
